In [8]:
import pandas as pd
import re

# =========================
# ĐỌC FILE EXCEL
# =========================

file_path = "data_chatbot_ntu.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="Thống kê các câu trả lời",
    header=3
)

QUESTION_COL = df.columns[1]

print(f"Tổng số dòng ban đầu: {len(df)}")

# ==================================================
# GIAI ĐOẠN 1: LÀM SẠCH DỮ LIỆU
# ==================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Xóa xuống dòng và tab
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # Xóa URL
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # Xóa email
    text = re.sub(r"\S+@\S+", "", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_question"] = df[QUESTION_COL].apply(clean_text)

# Xóa dòng rỗng
df_clean = df[
    df["clean_question"].str.strip() != ""
].copy()

# Xóa trùng hoàn toàn
df_clean = df_clean.drop_duplicates(
    subset=["clean_question"],
    keep="first"
)

print(f"Sau làm sạch: {len(df_clean)} dòng")

df_clean.to_excel(
    "01_data_after_cleaning.xlsx",
    index=False
)

print("Đã lưu: 01_data_after_cleaning.xlsx")

# ==================================================
# GIAI ĐOẠN 2: CHUẨN HÓA DỮ LIỆU
# ==================================================

ABBREVIATIONS = {
    "e": "",
    "k": "không",
    "ko": "không",
    "khong": "không",
    "dc": "được",
    "đc": "được",
    "el": "elearning",
    "sv": "sinh viên"
}

VIETNAMESE_DICT = {
    "dang": "đăng",
    "nhap": "nhập",
    "vao": "vào",
    "diem": "điểm",
    "o": "ở",
    "lich": "lịch",
    "khi": "khi",
    "nao": "nào",
    "hoc": "học",
    "phi": "phí",
    "mon": "môn",
    "nganh": "ngành",
    "truong": "trường",
    "sinh": "sinh",
    "vien": "viên",
    "tai": "tại",
    "sao": "sao",
    "duoc": "được",
    "khong": "không",
    "kiem": "kiểm",
    "tra": "tra",
    "ket": "kết",
    "qua": "quả"
}


def normalize_text(text):
    text = str(text).lower()

    # Xóa dấu câu
    text = re.sub(r"[^\w\s]", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    normalized_words = []

    for word in words:

        # Thay thế từ viết tắt
        if word in ABBREVIATIONS:
            replacement = ABBREVIATIONS[word]

            if replacement:
                normalized_words.extend(replacement.split())

            continue

        # Chuyển từ không dấu sang có dấu
        if word in VIETNAMESE_DICT:
            normalized_words.append(VIETNAMESE_DICT[word])
        else:
            normalized_words.append(word)

    return " ".join(normalized_words)


df_clean["normalized_question"] = (
    df_clean["clean_question"]
    .apply(normalize_text)
)

# Lưu các câu trùng sau chuẩn hóa
duplicates = df_clean[
    df_clean.duplicated(
        subset=["normalized_question"],
        keep=False
    )
].copy()

duplicates.to_excel(
    "02_duplicate_after_normalization.xlsx",
    index=False
)

print(f"Số dòng trùng sau chuẩn hóa: {len(duplicates)}")

# Xóa trùng sau chuẩn hóa
df_final = df_clean.drop_duplicates(
    subset=["normalized_question"],
    keep="first"
)

print(f"Sau chuẩn hóa: {len(df_final)} dòng")

df_final.to_excel(
    "03_final_dataset.xlsx",
    index=False
)

print("Đã lưu: 03_final_dataset.xlsx")

# ==================================================
# THỐNG KÊ
# ==================================================

print("\n========== THỐNG KÊ ==========")
print(f"Ban đầu: {len(df)}")
print(f"Sau làm sạch: {len(df_clean)}")
print(f"Sau chuẩn hóa: {len(df_final)}")
print("================================")

Tổng số dòng ban đầu: 1798
Sau làm sạch: 1590 dòng
Đã lưu: 01_data_after_cleaning.xlsx
Số dòng trùng sau chuẩn hóa: 24
Sau chuẩn hóa: 1577 dòng
Đã lưu: 03_final_dataset.xlsx

========== THỐNG KÊ ==========
Ban đầu: 1798
Sau làm sạch: 1590
Sau chuẩn hóa: 1577
